In [ ]:
%load_ext autoreload
%autoreload 

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime
from functools import partial
import gc

# local imports
import sys
sys.path.append('../../../')
from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.CutMasks.CutMasks import *

from makedf.mcstat import get_MCstat_unc

from analysis_village.cc1pi.var_configs import *

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
save_result = True
save_fig = save_result
save_fig_dir = "/exp/sbnd/data/users/lpelegri/Graphs/syst/detector"

if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)

# Load df

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')

#Load data
keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
data_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_data_rollingdev_bnblight.df", keys2load, 100)
data_evt_df = data_df['cc1pi']
data_hdr_df = data_df['hdr']

# BNB data
# data_tot_pot = data_hdr_df['TOR875'].sum()
data_tot_pot = data_hdr_df['pot'].sum()
print("data_tot_pot: %.3e" %(data_tot_pot))
pot_str = f"{data_tot_pot} $\\times 10^{18}$"
data_evt_df[pot_weight_col] = np.ones(len(data_evt_df))
data_gates = data_hdr_df.nbnbinfo.sum()
print("data tot gates : %.3e" %(data_gates))


In [ ]:

#Load systematics df
#syst_names = ["2xSCE", "PMTGainFluct","PMTHighNoise", "PMTLowEff", "CCalVar", "$C_{cal}$ Variation"]

syst_keys = ["SystVarsCV","wiremod_YZ","wiremod_XZ_thetaXW", "0xSCE","2xSCE","PMTGainFluct", "PMTHighNoise", "PMTLowEff", "ccalm", "ccalp", "alpham", "alphap", "rm", "rp", "betam", "betap"]
colors = ["black", "C0","C1","C2","C3","C4","C5","C6","C7","C8","C9","C10","C11","C12","C13","C14","C15","C16"]
labels    = ["CV", r"Wiremod Y-Z",r"Wiremod X$\theta_{xw}$", "0xSCE", "2xSCE","PMTGainFluct", "PMTHighNoise", "PMTLowEff", "ccalm", "ccalp", "alpham", "alphap", "rm", "rp", "betam", "betap"]
'''
syst_keys = ["SystVarsCV","wiremod_YZ", "ccalm", "ccalp"]
colors = ["black", "C0","C1","C2","C3","C4","C5","C6","C7","C8","C9","C10","C11","C12","C13","C14","C15","C16"]
labels    = ["CV", r"Wiremod Y-Z", "ccalm", "ccalp"]
'''
detvar_plotter = partial(
    variation_hists,
    var_colors=colors,
    var_labels=labels,
    approval="internal"
)

detvar_plotter_final_vars = partial(
    variation_hists_final_vars,
    var_colors=colors,
    var_labels=labels,
    approval="internal"
)

keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
syst_dfs = {}
for key in syst_keys:
    syst_df = load_df(f"/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_{key}.df",keys2load,100)
        
    syst_tot_pot = syst_df['hdr']['pot'].sum()
    print("syst_pot: %.3e" %(syst_tot_pot))
    syst_pot_scale = data_tot_pot / syst_tot_pot
    syst_df['cc1pi'][pot_weight_col] = syst_pot_scale * np.ones(len(syst_df['cc1pi']))

    syst_dfs[key] = syst_df

In [ ]:
for key, df in syst_dfs.items():
    df['cc1pi'] = perform_truth_matching(df['cc1pi'], df['nudf'])

In [ ]:
#CV_evt_df = CV_df['cc1pi']
syst_evt_dfs = {}
for key, df in syst_dfs.items():
    syst_evt_dfs[key] = df['cc1pi']
    syst_evt_dfs[key][('slc', 'cut', 'proton_BDT_sideband', '', '', '')] = proton_BDT_sideband_mask(syst_evt_dfs[key], ['__ntuple', 'entry', 'rec.slc..index'])

del syst_dfs
gc.collect()

In [ ]:
# --- CV ---
#CV_masks = build_event_masks(CV_evt_df)
plot_sideband = True
syst_masks = {}
for key, df in syst_evt_dfs.items():
    masks = build_event_cumulative_masks(df, plot_sideband = plot_sideband)
    syst_masks[key] = masks

In [ ]:
for key in syst_evt_dfs:
    syst_evt_dfs[key] = syst_evt_dfs[key][syst_masks[key]["energy"]]

del syst_masks
gc.collect()

# Get all the syst uncert

In [ ]:
'''
var_config = VariableConfig.pion_momentum()
slice_levels = ['__ntuple', 'entry', 'rec.slc..index']
evtdfs = [syst_evt_dfs[syst_key][syst_evt_dfs[syst_key].truth.nu_categ == "CC1pi"].groupby(level=slice_levels, sort=False).first() for syst_key in syst_evt_dfs.keys()]


var_name = var_config.var_evt_reco_col
bins = var_config.bins
pot_label = f"Candidate Slices (POT={pot_str})"
plot_labels = [var_config.var_labels[1], pot_label, ""]
approval = "internal"
save_name = save_fig_dir + "/{}.png".format(var_config.var_save_name)
n = detvar_plotter_final_vars(evtdfs, 
                    var_name=var_name,
                    bins=bins,
                    plot_labels=plot_labels,
                    approval=approval,
                    vline=[],
                    save_fig=save_fig, save_name=save_name)
'''

In [ ]:
'''
ret_dict = {}
for kidx, syst_key in enumerate(syst_dfs.keys()):
    if syst_key == "SystVarsCV":
        continue

    # take syst variation as a unisim uncertainty
    cv_events = n[0]
    univ_events = np.array([n[kidx]]) 
    ret = get_covariance_matrix(univ_events, cv_events)
    ret_dict[syst_key] = ret
    plot_univ_hists(univ_events, 
                    cv_events,
                    syst_key, 
                    var_config,
                    use_bin_width = False,
                    )

    matrix_type = "cov_frac"
    plot_labels = [var_config.var_labels[2], var_config.var_labels[1], ""]
    save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_key, matrix_type)
    title = "{} {}".format(syst_key, matrix_type)
    plot_heatmap(ret[matrix_type], 
                 bins=bins,
                 plot_labels=plot_labels,
                 save_fig=save_fig, 
                 save_name=save_fig_name)

    frac_unc = np.sqrt(np.diag(ret["cov_frac"]))
    plot_frac_unc([(frac_unc, syst_key)], var_config)
'''

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

# --- 1. Pretty Label Mapping ---
# Maps raw systematic keys to publication-quality LaTeX labels
pretty_label_map = {
    "CV": "Nominal (CV)",
    "wiremod_YZ": r"WireMod $Y$-$Z$",
    "wiremod_XZ_thetaXW": r"WireMod $X\theta_{xw}$",
    "0xSCE": "0xSCE",
    "2xSCE": "2xSCE",
    "PMTGainFluct": "PMT Gain Fluctuation",
    "PMTHighNoise": "PMT High Noise",
    "PMTLowEff": "PMT Low Efficiency",
    "phi": r"Recomb. $\phi$ model",
    "ccal": r"Recomb. $C_{cal}$",
    "alpha": r"Recomb. $\alpha$",
    "r": r"Recomb. $R$",
    "beta": r"Recomb. $\beta_{90}$"
}# --- 2. Configuration ---

unisim_keys = []
paired_syst = {} 
file_dir = "/exp/sbnd/data/users/lpelegri/syst/frac_cov_matrices"
os.makedirs(file_dir, exist_ok=True)

if 'syst_dict' not in locals():
    syst_dict = {}

var_configs = [
    VariableConfig.all_evts(), 
    VariableConfig.muon_momentum(),
    VariableConfig.muon_direction(),
    VariableConfig.pion_momentum(),
    VariableConfig.pion_direction(),
    VariableConfig.angle_between_candidates(),
    VariableConfig.num_protons(),
    VariableConfig.delta_pt(),
    VariableConfig.delta_alpha_T(),
    VariableConfig.delta_phi_T()
]

# Identify Unisim (1 univ) vs Paired/Multisim (2 univ)
for key in syst_keys:
    if key == "SystVarsCV": continue
    if key.endswith('m') or key.endswith('p'):
        base = key[:-1]
        if base not in paired_syst:
            paired_syst[base] = [None, None]
        if key.endswith('m'): paired_syst[base][0] = key
        else: paired_syst[base][1] = key
    else:
        unisim_keys.append(key)

# --- 3. Main Variable Loop ---
for var_config in var_configs:
    evtdfs = [syst_evt_dfs[syst_key].groupby(level=['__ntuple', 'entry', 'rec.slc..index'], sort=False).first() for syst_key in syst_evt_dfs.keys()]
    
    n = detvar_plotter_final_vars(evtdfs, var_name=var_config.var_evt_reco_col, bins=var_config.bins, plot=False)

    key_to_idx = {key: i for i, key in enumerate(syst_evt_dfs.keys())}
    cv_events = n[0]
    ret_dict = {}

    print(cv_events)
    # Covariance Calculations
    for syst_key in unisim_keys:
        kidx = key_to_idx[syst_key]
        ret_dict[syst_key] = get_covariance_matrix(np.array([n[kidx]]), cv_events)
        print(n[kidx])
        
    for base_name, keys in paired_syst.items():
        m_idx, p_idx = key_to_idx.get(keys[0]), key_to_idx.get(keys[1])
        if m_idx is not None and p_idx is not None:
            ret_dict[base_name] = get_covariance_matrix(np.array([n[m_idx], n[p_idx]]), cv_events)

    # Aggregate Total Uncertainty
    total_cov_frac = None
    total_cov = None
    uncertanties_list = []
    
    for syst_key, ret in ret_dict.items():
        frac_unc = np.sqrt(np.diag(ret["cov_frac"]))
        uncertanties_list.append((frac_unc, syst_key))
        
        if total_cov_frac is None: total_cov_frac = ret["cov_frac"].copy()
        else: total_cov_frac += ret["cov_frac"]
            
        if total_cov is None: total_cov = ret["cov"].copy()
        else: total_cov += ret["cov"]

    
    total_corr = corr_from_fraccov(total_cov)

    f_name = f"detector_{var_config.var_save_name}_frac_cov.pdf" 
    save_full_path = os.path.join(save_fig_dir, f_name)
       
    plot_heatmap(total_cov_frac, 
            var_config.bins, 
            plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Fractional Covariance"],
            save_fig=save_fig, save_name=save_full_path)

    f_name = f"detector_{var_config.var_save_name}_cov.pdf" 
    save_full_path = os.path.join(save_fig_dir, f_name)
    plot_heatmap(total_cov, 
            var_config.bins, 
            plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Covariance"],
            save_fig=save_fig, save_name=save_full_path)
        
    f_name = f"detector_{var_config.var_save_name}_corr.pdf" 
    save_full_path = os.path.join(save_fig_dir, f_name)
    plot_heatmap(total_corr, 
            var_config.bins, 
            plot_labels=[var_config.var_labels[1], var_config.var_labels[1], "Correlation"],
            save_fig=save_fig, save_name=save_full_path)

    # --- 4. Plotting ---
    fig, ax = plt.subplots(figsize=(8, 6))
    total_values = np.sqrt(np.diag(total_cov_frac)) * 100 

    # SORT BY IMPORTANCE (Integral of uncertainty)
    sorted_uncertanties = sorted(uncertanties_list, key=lambda x: np.sum(x[0]), reverse=True)
    
    for frac_unc, syst_key in sorted_uncertanties:
        display_label = pretty_label_map.get(syst_key, syst_key.replace("_", " "))
        ax.hist(
            var_config.bin_centers,
            bins=var_config.bins,
            weights=frac_unc * 1e2,
            histtype="step",
            linewidth=2,
            label=display_label,
            zorder=3
        )

    ax.hist(
        var_config.bin_centers,
        bins=var_config.bins,
        weights=total_values,
        histtype="step",
        linewidth=2,
        color="k",
        label="Total",
        zorder=15  # Ensure the total is always on top
    )
    
    # Formatting
    ax.legend(loc="upper center", ncol=3, fontsize=12, frameon=True, edgecolor='gray')
    ax.set_xlim(var_config.bins[0], var_config.bins[-1])
    ax.set_ylim(0, max(total_values) * 1.6) 
    ax.set_xlabel(var_config.var_labels[1])
    ax.set_ylabel("Uncertainty [%]")
    
    ax.grid(which='major', linestyle='-', linewidth=0.7, alpha=0.7)
    ax.grid(which='minor', linestyle=':', linewidth=0.5, alpha=0.5)
    ax.minorticks_on()
    
    plt.tight_layout()
    plt.savefig(os.path.join(save_fig_dir, f"detvar_{var_config.var_save_name}.png"), dpi=300)
    plt.show()

    # Store for NPZ
    syst_dict[var_config.var_save_name] = total_cov_frac
    for key, ret in ret_dict.items():
        syst_dict[f"{var_config.var_save_name}_{key}"] = ret["cov_frac"]

# Final Save
if save_result:
    np.savez(os.path.join(file_dir, "detvar_syst_dict.npz"), **syst_dict)

In [ ]:
'''The 
file_dir = "/exp/sbnd/data/users/lpelegri/syst/frac_cov_matrices"
if 'syst_dict' not in locals():
    syst_dict = {}
os.makedirs(file_dir, exist_ok=True)  # create directory if needed

var_configs = [
    VariableConfig.all_evts(), 
    VariableConfig.muon_momentum(),
    VariableConfig.muon_direction(),
    VariableConfig.pion_momentum(),
    VariableConfig.pion_direction(),
    VariableConfig.angle_between_candidates(),
    VariableConfig.num_protons(),
    VariableConfig.delta_pt(),
    VariableConfig.delta_alpha_T(),
    VariableConfig.delta_phi_T()
]
syst_name = "detvar"

for var_config in var_configs:
    evtdfs = [syst_evt_dfs[syst_key].groupby(level=slice_levels, sort=False).first() for syst_key in syst_evt_dfs.keys()]
    var_name = var_config.var_evt_reco_col
    bins = var_config.bins
    pot_label = f"Candidate Slices (POT={pot_str})"
    plot_labels = [var_config.var_labels[1], pot_label, ""]
    approval = "internal"
    save_name = save_fig_dir + "/{}.png".format(var_config.var_save_name)
    n = detvar_plotter_final_vars(evtdfs, 
                        var_name=var_name,
                        bins=bins,
                        plot_labels=plot_labels,
                        approval=approval,
                        plot = True
                    )

    ret_dict = {}
    for kidx, syst_key in enumerate(syst_dfs.keys()):
        if syst_key == "SystVarsCV":
            continue
    
        # take syst variation as a unisim uncertainty
        cv_events = n[0]
        univ_events = np.array([n[kidx]]) 
        ret = get_covariance_matrix(univ_events, cv_events)
        ret_dict[syst_key] = ret
        '''
        plot_univ_hists(univ_events, 
                        cv_events,
                        syst_name, 
                        var_config,
                        use_bin_width = False,
                        )
    
        matrix_type = "cov_frac"
        plot_labels = [var_config.var_labels[2], var_config.var_labels[1], ""]
        save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
        title = "{} {}".format(syst_key, matrix_type)

        plot_heatmap(ret[matrix_type], 
                     bins=bins,
                     plot_labels=plot_labels,
                     save_fig=save_fig, 
                     save_name=save_fig_name)
        frac_unc = np.sqrt(np.diag(ret["cov_frac"]))
        plot_frac_unc([(frac_unc, syst_key)], var_config)

        '''
        
        
    # get total detector variation covariance matrix
    uncertanties_list = []
    for kidx, syst_key in enumerate(ret_dict.keys()):
        uncertanties_list.append((np.sqrt(np.diag(ret_dict[syst_key]["cov_frac"]))*100, syst_key))
        if kidx == 0:
            detvar_total_cov = ret_dict[syst_key]["cov_frac"]
        else:
            detvar_total_cov += ret_dict[syst_key]["cov_frac"]
        

    fig, ax = plt.subplots(figsize=(10,6))

    box = ax.get_position()
    ax.set_position([box.x0, box.y0, box.width * 0.8, box.height])

    handles = []
    labels = []
    
    for syst, name in uncertanties_list:
    
        h = ax.hist(
            var_config.bin_centers,
            bins=var_config.bins,
            weights=syst,
            histtype="step",
            linewidth=2,
            label=name
        )

        handles.append(h[2][0])
        labels.append(name)

    # ---- Total ----
    frac_uncert_total = np.sqrt(np.diag(detvar_total_cov))
    total_values = frac_uncert_total * 1e2

    total_handle = ax.hist(
        var_config.bin_centers,
        bins=var_config.bins,
        weights=total_values,
        histtype="step",
        linewidth=3,
        color="k",
        label="Total"
    )[2][0]
    
    handles.append(total_handle)
    labels.append("Total")
    
    # Legend
    ax.legend(
        handles,
        labels,
        loc="upper center", 
        ncol=3, 
        fontsize=11
    )

    ax.set_xlim(var_config.bins[0], var_config.bins[-1])
    ax.set_ylim(0, max(total_values) * 1.4)
    ax.set_xlabel(var_config.var_labels[1])
    ax.set_ylabel("Uncertainty [%]")

    ax.grid(which='major', linestyle='-', linewidth=0.7, alpha=0.7)
    ax.grid(which='minor', linestyle=':', linewidth=0.5, alpha=0.5)
    ax.minorticks_on()

    syst_dict[var_config.var_save_name] = detvar_total_cov
    for syst_key in ret_dict.keys():
        syst_dict[var_config.var_save_name + "_" + syst_key] = ret_dict[syst_key]["cov_frac"]
    
# save syst_dict as an npz file in the directory where dfs were loaded from
if save_result:
    print("saving syst_dict as npz in %s" % (file_dir))
    np.savez(file_dir + "/detvar_syst_dict.npz", **syst_dict)
'''